# 10.5 Übungen

## Übung 10.5 (✩)

Ein Raum kühlt nachts gegen die kalte Außenluft ab. Um 06:00 Uhr (im
Modell $t = 10\,\text{min}$) springt der Thermostat an und erhöht die
Zieltemperatur. Die Raumtemperatur folgt der DGL

$$\dot{T}(t) = -k\bigl(T(t) - T_\infty(t)\bigr), \qquad
T_\infty(t) = \begin{cases} 16\,°C & t < 10\,\text{min} \\ 22\,°C & t \geq 10\,\text{min} \end{cases}$$

mit $k = 0.15\,\text{min}^{-1}$ und Anfangstemperatur $T_0 = 26\,°C$.

```python
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('seaborn-v0_8')

k     = 0.15    # Abkühlkonstante in 1/min
T0    = 26.0    # Anfangstemperatur in °C
t_end = 30.0    # Simulationsende in min

def T_inf_thermostat(t):
    # Zieltemperatur des Thermostats: nachts kälter, ab t=10 min wärmer
    if t < 10.0:
        return 16.0    # Nacht: Heizung aus
    else:
        return 22.0    # Tag: Heizung ein

def f_thermostat(t, y):
    return [-k * (y[0] - T_inf_thermostat(t))]

t_eval = np.linspace(0, t_end, 601)
sol    = solve_ivp(f_thermostat, (0, t_end), [T0],
                   t_eval=t_eval, max_step=0.1)

print(f"sol.success:  {sol.success}")
print(f"T(0 min)  = {sol.y[0,   0]:.4f} °C")
print(f"T(10 min) = {sol.y[0, 200]:.4f} °C")
print(f"T(15 min) = {sol.y[0, 300]:.4f} °C")
print(f"T(30 min) = {sol.y[0,  -1]:.4f} °C")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sol.t, sol.y[0], color='#005A94', linewidth=2, label='$T(t)$')
ax.axhline(16.0, color='#CCDEE9', linewidth=1.2, linestyle=':',
           label='$T_\\infty = 16\\,°C$ (Nacht)')
ax.axhline(22.0, color='#E87846', linewidth=1.2, linestyle=':',
           label='$T_\\infty = 22\\,°C$ (Tag)')
ax.axvline(10.0, color='#484949', linewidth=1.0, linestyle='--',
           label='Thermostat-Schaltpunkt')
ax.set_xlabel('Zeit in min')
ax.set_ylabel('Temperatur in °C')
ax.set_title('Raumtemperatur mit Thermostat-Schaltung')
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()
```

1. Zeichnen Sie $T_\infty(t)$ für $t \in [0, 30]$ min von Hand als Skizze.
   Um was für eine Funktion handelt es sich?
2. Ist $T(t)$ bei $t = 10\,\text{min}$ stetig? Ist $\dot{T}(t)$ stetig?
   Begründen Sie ohne Code.
3. Berechnen Sie $\dot{T}(0)$ von Hand. Berechnen Sie anschließend das
   Vorzeichen von $\dot{T}(15\,\text{min})$, indem Sie $T(15) \approx 20\,°C$
   als Näherung verwenden. Was bedeuten die Vorzeichen physikalisch?
4. Wo hat $T(t)$ ein Minimum, und warum tritt es genau dort auf?
   Erklären Sie in einem Satz ohne Code.
5. Führen Sie den Code aus und überprüfen Sie Ihre Vorhersagen.

In [ ]:
# Code-Zelle

## Übung 10.6 (✩✩)

Ein Kondensator $C = 100\,\mu F$ wird über einen Widerstand $R = 1000\,\Omega$
an eine Spannungsquelle $U_0 = 12\,V$ angeschlossen. Die Spannung am
Kondensator $U_C$ genügt der DGL

$$\dot{U}_C = \frac{U_0(t) - U_C}{\tau}, \qquad \tau = RC.$$

Bei $t = 0$ ist der Kondensator ungeladen ($U_C(0) = 0$). Nach einer
Zeitkonstante ($t = \tau$) wird die Spannungsquelle abgeschaltet
($U_0 \to 0$), sodass der Kondensator über denselben Widerstand entlädt.

```python
import numpy as np

U0  = 12.0         # Quellspannung in V
R   = 1000.0       # Widerstand in Ohm
C   = 100e-6       # Kapazität in F
tau = R * C        # Zeitkonstante in s
```

1. Implementieren Sie `f_rc_laden(t, y)` für den reinen Ladevorgang
   ($U_0 = 12\,V$, autonom) und lösen Sie die DGL für $5\tau$ Sekunden.
   Stellen Sie $U_C(t)$ dar. Zeichnen Sie $\tau$ auf der $t$-Achse und
   $U_0(1 - 1/e) \approx 7.58\,V$ auf der $y$-Achse als Hilfslinie ein.
2. Bestimmen Sie numerisch mit `np.searchsorted`, wann $U_C$ erstmals
   99 % von $U_0$ erreicht. Vergleichen Sie mit der analytischen Formel
   $t_{99} = -\tau\,\ln(0.01)$.
3. Implementieren Sie `f_rc_nichtautonom(t, y)` als nichtautonome DGL:
   $U_0(t) = 12\,V$ für $t < \tau$, sonst $U_0(t) = 0$. Simulieren Sie
   für $4\tau$ Sekunden und stellen Sie Lade- und Entladevorgang in
   einem gemeinsamen Plot dar.
4. Lesen Sie die Zeitkonstante aus dem Entladeverlauf ab: Bei $t = \tau$
   nach dem Abschalten (also bei $t = 2\tau$ insgesamt) sollte
   $U_C(2\tau) = U_C(\tau)/e$ gelten. Prüfen Sie numerisch.

In [ ]:
# Code-Zelle

## Übung 10.7 (✩✩✩)

Ein Fallschirmspringer springt aus dem Flugzeug und öffnet den Schirm
noch nicht. Neben der Schwerkraft wirkt eine quadratische Luftreibung:

$$\dot{v}(t) = g - k_q\,v(t)^2, \qquad v(0) = 0.$$

Die Endgeschwindigkeit (Gleichgewicht $\dot{v} = 0$) ist
$v_\infty = \sqrt{g/k_q}$. Die analytische Lösung lautet
$v(t) = v_\infty \tanh\!\bigl(g\,t/v_\infty\bigr)$.

```python
import numpy as np

g    = 9.81    # Schwerebeschleunigung in m/s^2
k_q  = 0.006   # spezifischer Luftwiderstandsbeiwert in 1/m
```

**Teil 1 – Geschwindigkeitsverlauf**

1. Implementieren Sie `f_fall_quad(t, y)` und lösen Sie die DGL für 60 s.
   Stellen Sie $v(t)$ dar und markieren Sie $v_\infty$ als waagerechte
   Linie. Berechnen Sie $v_\infty = \sqrt{g/k_q}$ und vergleichen Sie den
   numerischen Endwert $v(60\,\text{s})$ damit.
2. Überlagern Sie die analytische Lösung
   $v_\text{ana}(t) = v_\infty\,\tanh(g\,t/v_\infty)$ im selben Plot.
   Berechnen Sie den maximalen absoluten Fehler zwischen `solve_ivp` und
   der analytischen Lösung.

**Teil 2 – Parameterstudie**

3. Erstellen Sie eine Kurvenschar: Lösen Sie die DGL für vier Werte
   $k_q \in \{0.003,\, 0.006,\, 0.012,\, 0.024\}\,\text{m}^{-1}$ und
   stellen Sie alle vier Kurven in einem Plot dar. Geben Sie die
   zugehörigen Endgeschwindigkeiten als Tabelle aus.

**Teil 3 – Zurückgelegte Strecke**

4. Berechnen Sie die zurückgelegte Strecke $h(t) = \int_0^t v(\xi)\,\mathrm{d}\xi$
   mit `scipy.integrate.cumulative_trapezoid` auf der `solve_ivp`-Ausgabe
   (kein neues Gleichungssystem nötig). Stellen Sie $h(t)$ in einem zweiten
   Plot dar. Lesen Sie ab: Nach wie vielen Sekunden hat der Springer
   1000 m zurückgelegt?

**Teil 4 – Vergleich mit linearem Luftwiderstand**

5. Der lineare Luftwiderstand führt auf $\dot{v} = g - k_l\,v$ mit
   $v_\infty = g/k_l$. Wählen Sie $k_l = g/v_\infty$ so, dass beide
   Modelle dieselbe Endgeschwindigkeit haben, und lösen Sie beide DGLen.
   Plotten Sie $v_\text{lin}(t)$ und $v_\text{quad}(t)$ gemeinsam.
   Welches Modell erreicht $v_\infty$ schneller? Berechnen Sie
   $v(\tau)/v_\infty$ für beide Modelle bei $\tau = v_\infty/g$ und
   erklären Sie den Unterschied in zwei Sätzen.

In [ ]:
# Code-Zelle